In [ ]:
! pip install transformers datasets
! pip install evaluate
! pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 13.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system ==

### Προσοχή

Μη διαγράψετε τα # insert your code here σχόλια, καθώς βοηθούν στη διόρθωση. Συμπληρώστε τον κώδικά σας μετά από τα σχόλια αυτά.

# Μέρος Α: Fine-tune a pretrained model

Τα γλωσσικά μοντέλα αποτελούνται από δύο στάδια εκπαίδευσης:
1. **Pre-training σε μεγάλα unlabelled datasets**:

  Το pre-training είναι υπολογιστικά πολύ ακριβό και γι αυτό στην πράξη δε το χρησιμοποιούμε όταν θέλουμε να τρέξουμε ένα μοντέλο σε ένα καινούργιο dataset. Μπορούμε να σκεφτούμε το pre-training ως τη διαδικασία εκμάθησης γλωσσικών κανόνων κι εννοιών, οι οποίες στη συνέχεια μπορούν να χρησιμοποιηθούν για διάφορους σκοπούς.

2. **Fine-tuning σε μικρότερα labelled datasets**:
  
     Το fine-tuning πρακτικά εκμεταλλεύεται τις ιδιότητες του transfer learning προκειμένου να μεταφέρουμε τη 'γνώση' που έχει αποθηκευθεί στο γλωσσικό μοντέλο κατά τη διάρκεια του pre-training σε συγκεκριμένα task. Κάθε task εξυπηρετείται μέσω στοχευμένων datasets. Για παράδειγμα, κάποια datasets αναφέρονται στην ταξινόμηση κειμένων σε κατηγιορίες (text classification), άλλα datasets περιέχουν ερωτήσεις οι οποίες πρέπει να απαντηθούν (question answering) κι άλλα πολλά.

Κάποια κλασικά tasks της επεξεργασίας φυσικής γλώσσας είναι τα ακόλουθα:
- Text classification
- Question answering
- Natural language inference
- Fill mask
- Semantic similarity

Περισσότερες πληροφορίες μπορείτε να βρείτε στον ακόλουθο σύνδεσμο στο domain Natural Language Processing: https://huggingface.co/models

Στο πρώτο κομμάτι της παρούσας εργαστηριακής άσκησης, θα χρησιμοποιήσουμε το pre-training fine-tuning σενάριο για να ταξινομήσουμε reviews.

## Pipelines

Με τη χρήση του **text-classification pipeline** μπορούμε να τρέξουμε γλωσσικά μοντέλα που αφορούν tasks ταξινόμησης.

Το natural language inference (NLI) task αποτελεί ένα task ταξινόμησης, αφού το σχετικό μοντέλο (εν προκειμένω το roberta-large-mnli) καλείται να ταξινομήσει ένα κείμενο σε μία από τις 3 κατηγορίες: **[entailment/neutral/contradiction]**.

```
from transformers import pipeline

classifier = pipeline("text-classification", model = "roberta-large-mnli")
classifier("A soccer game with multiple males playing. Some men are playing a sport.")
## [{'label': 'ENTAILMENT', 'score': 0.98}]
```

Ένα άλλο task ταξινόμησης αφορά την αξιολόγηση του κατά πόσο ένα κείμενο είναι **γραμματικά ορθό (acceptable) ή όχι (unacceptable)**:

```
from transformers import pipeline

classifier = pipeline("text-classification", model = "textattack/distilbert-base-uncased-CoLA")
classifier("I will walk to home when I went through the bus.")
##  [{'label': 'unacceptable', 'score': 0.95}]
```

## Σύνολο δεδομένων Yelp polarity

Κατεβάζουμε το [Yelp Polarity](https://huggingface.co/datasets/yelp_polarity) dataset το οποίο περιέχει reviews που εκφράζουν συναισθήματα πελατών για εστιατόρια.
Το  Yelp κατασκευάστηκε θεωρώντας τα αστέρια 1 και 2 αρνητικά και τα 3 και 4 θετικά.  Η αρνητική πολικότητα ανήκει στην κατηγορία 1 και η θετική στην κατηγορία 2. Τα reviews αυτά χωρίζονται σε αυτές τις κατηγορίες, και ο σκοπός μας είναι να κατηγοριοποιήσουμε νέα reviews στις σωστές κατηγορίες.



In [ ]:
from datasets import load_dataset

# insert your code here
dataset = load_dataset("yelp_polarity")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/8.93k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/256M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/560000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/38000 [00:00<?, ? examples/s]

In [ ]:
# Print dataset splits
print("Available splits:", dataset.keys())

# Show the number of examples in each split
print("Number of training examples:", len(dataset["train"]))
print("Number of test examples:", len(dataset["test"]))

# Display a sample from the training set
print("\nSample example:")
print(dataset["train"][0])


Available splits: dict_keys(['train', 'test'])
Number of training examples: 560000
Number of test examples: 38000

Sample example:
{'text': "Unfortunately, the frustration of being Dr. Goldberg's patient is a repeat of the experience I've had with so many other doctors in NYC -- good doctor, terrible staff.  It seems that his staff simply never answers the phone.  It usually takes 2 hours of repeated calling to get an answer.  Who has time for that or wants to deal with it?  I have run into this problem with many other doctors and I just don't get it.  You have office workers, you have patients with medical needs, why isn't anyone answering the phone?  It's incomprehensible and not work the aggravation.  It's with regret that I feel that I have to give Dr. Goldberg 2 stars.", 'label': 0}


Επειδή το σύνολο δεδομένων του Yelp Polarity περιέχει πολλά δείγματα, προκειμένου να επιταχύνουμε τη διαδικασία του fine-tuning συστήνουμε να διατηρήσετε 300 δείγματα από το train set και 300 δείγματα από το test set.

Ελέγξτε τον αριθμό κατηγοριών που υπάρχουν συνολικά στο train και το test set και διατηρήστε ισορροπημένο αριθμό δειγμάτων ανά κατηγορία για τα σύνολα αυτά κατά την επιλογή των 300 δειγμάτων.

In [ ]:
# insert your code here
from datasets import concatenate_datasets

# Filter 150 examples per class for both train and test sets (total 300 each)
def get_balanced_subset(split, n_per_class=150):
    label_0 = dataset[split].filter(lambda x: x["label"] == 0).select(range(n_per_class))
    label_1 = dataset[split].filter(lambda x: x["label"] == 1).select(range(n_per_class))
    return concatenate_datasets([label_0, label_1])

train_dataset = get_balanced_subset("train", n_per_class=150)
test_dataset = get_balanced_subset("test", n_per_class=150)

# Check final dataset sizes
print("\nFiltered train size:", len(train_dataset))
print("Filtered test size:", len(test_dataset))

# Check label distribution
from collections import Counter
print("\nTrain label distribution:", Counter(train_dataset["label"]))
print("Test label distribution:", Counter(test_dataset["label"]))


Filter:   0%|          | 0/560000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/560000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/38000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/38000 [00:00<?, ? examples/s]


Filtered train size: 300
Filtered test size: 300

Train label distribution: Counter({0: 150, 1: 150})
Test label distribution: Counter({0: 150, 1: 150})


In [ ]:
train_dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 300
})

# Language Models

Η προεπεξεργασία των κειμένων προηγείται της εισόδου τους στα γλωσσικά μοντέλα.

Η διαδικασία αυτή επιτελείται μέσω των **Tokenizers**, τα οποία μετατρέπουν τα tokens εισόδου σε κατάλληλα IDs του λεξιλογίου προεκπαίδευσης, κι έτσι μετατρέπουν το κείμενο σε μορφή που μπορεί να επεξεργαστεί κάποιο μοντέλο Transformer. Η βιβλιοθήκη Huggingface προσφέρει εύκολες και high-level υλοποιήσεις tokenization, τις οποίες συστήνουμε να ακολουθήσετε στη συνέχεια.

Συγκεκριμένα, **αρχικοποιούμε τη διαδικασία του tokenization με χρήση του AutoTokenizer**. Επιλέγοντας τη μέθοδο **from_pretrained** λαμβάνουμε έναν tokenizer που αποκρίνεται στην αρχιτεκτονική του μοντέλου που επιθυμούμε να χρησιμοποιήσουμε, παρέχοντας συμβατό tokenization.

Περισσότερες πληροφορίες για το AutoTokenization μπορείτε να βρείτε εδώ:
https://huggingface.co/docs/transformers/model_doc/auto

Αναφορικά με το μοντέλο BERT, μπορείτε να δείτε τη διαδικασία [του tokenization και της αρχικοποίησης του μοντέλου](https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertTokenizer):

```
from transformers import AutoTokenizer, BertModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")
```

Στα πλαίσια της άσκησης καλείστε να επιτελέσετε την παραπάνω διαδικασία με *κάποιο άλλο μοντέλο της επιλογής σας από το Huggingface* που να υποστηρίζει τον AutoTokenizer. Το pre-trained μοντέλο που θα επιλέξετε θα πρέπει να διαθέτει υλοποίηση με sequence classification head (κατ αναλογία της μεθόδου BertForSequenceClassification).

Στο επόμενο κελί, φορτώστε το επιλεχθέν μοντέλο με τον αντίστοιχο tokenizer.

(Αγνοήστε πιθανά warnings της μορφής Some weights of the model checkpoint at xxx were not used when initializing...)

In [ ]:
# insert your code here
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load the tokenizer and model for sequence classification
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Σας παρέχουμε τη συνάρτηση που πραγματοποιεί το tokenization καλώντας τον tokenizer που επιλέξατε. Εφαρμόστε το τόσο στο train, όσο και στο test set.

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

# insert your code here
# Tokenize the datasets using the provided tokenize_function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

# Apply tokenization to both train and test sets
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Τυπώνοντας το train ή το test set, θα δείτε δύο επιπλέον πεδία 'input_ids' και 'attention_mask'. Βεβαιωθείτε ότι υπάρχουν, άρα και το tokenization έχει επιτευχθεί.

In [ ]:
tokenized_train

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 300
})

## Χρήση του PyTorch Trainer για fine-tuning

Η κλάση [Trainer](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer) έχει βελτιστοποιηθεί από τους δημιουργούς του Huggingface παρέχοντας πολλές διευκολύνσεις και λιγότερη 'χεράτη' δουλειά. Προτείνουμε να τη χρησιμοποιήσετε ως εναλλακτική του να γράψετε το δικό σας training loop.
Καθώς η Trainer δεν τεστάρει αυτόματα την επίδοση του εκάστοτε μοντέλου κατά την εκπαίδευση, παρέχουμε κατάλληλη συνάρτηση προκειμένου να αποτιμάται το accuracy του μοντέλου σε κάθε εποχή.

### Υλοποίηση

In [ ]:
import numpy as np
import evaluate
import torch
from tqdm import tqdm
from transformers import pipeline

# Load metric
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

Η κλάση [TrainingArguments](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.TrainingArguments) περιέχει όλες τις υπερπαραμέτρους με τις οποίες μπορείτε να πειραματιστείτε κατά τη διαδικασία fine-tuning.


Καλείστε να πειραματιστείτε με διαφορετικές υπερπαραμέτρους όπως το learning rate, batch size κλπ, καθώς επίσης και να ορίσετε optimizer και scheduler για το fine-tuning. Προτείνουμε να εκτελέσετε fine-tuning για μικρό αριθμό εποχών (άλλωστε το μοντέλο είναι ήδη προεκπαιδευμένο).

1. Θα μας δώσετε σε markdown ένα πινακάκι με διαφορετικές υπερπαραμέτρους που δοκιμάσατε και το accuracy που πετύχατε στην τελευταία εποχή.

2. Βάσει των πειραματισμών, πώς επηρεάζουν διαφορετικές υπερπαράμετροι όπως το learning rate και το batch size το fine-tuning του μοντέλου που επιλέξατε;  Σχολιάστε και αναλύστε.

In [ ]:
from transformers import TrainingArguments, Trainer, get_scheduler
from transformers import DataCollatorWithPadding
from torch.optim import AdamW

# insert your code here

# Run a single experiment and return results
def run_experiment(model, tokenizer, train_dataset, test_dataset, config, run_id=0):
    # TrainingArguments
    args = TrainingArguments(
        output_dir=f"trainer_run_{run_id}",
        eval_strategy="epoch",
        per_device_train_batch_size=config["batch_size"],
        per_device_eval_batch_size=config["batch_size"],
        num_train_epochs=config["epochs"],
        learning_rate=config["lr"],
        weight_decay=config.get("weight_decay", 0.01),
        report_to="none",
        logging_dir='./logs',
        logging_steps=10,
        save_strategy="no"
    )

    # Optimizer
    optimizer = AdamW(model.parameters(), lr=args.learning_rate)

    # Scheduler
    num_training_steps = len(train_dataset) // args.per_device_train_batch_size * args.num_train_epochs
    lr_scheduler = get_scheduler(
        name="linear",
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps
    )

    # Data collator (handles dynamic padding)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # Trainer
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
        data_collator=data_collator,
        optimizers=(optimizer, lr_scheduler)
    )

    # Train the model
    trainer.train()

    # Evaluate after training
    eval_result = trainer.evaluate()
    accuracy = eval_result.get("eval_accuracy", 0.0)

    # Return results
    return {
        "learning_rate": config["lr"],
        "batch_size": config["batch_size"],
        "epochs": config["epochs"],
        "accuracy": round(accuracy, 4)
    }


# Run a list of experiments
def run_experiments(model, tokenizer, train_dataset, test_dataset, experiment_configs):
    all_results = []
    for i, config in enumerate(experiment_configs):
        print(f"\nRunning experiment {i+1}/{len(experiment_configs)}: \n{config}")
        result = run_experiment(model, tokenizer, train_dataset, test_dataset, config, run_id=i)
        all_results.append(result)
    return all_results

def display_results_table(results):
    print("\nFinal Results Table:\n")

    # Header
    header = "| Learning Rate | Batch Size | Epochs | Accuracy |"
    divider = "|---------------|------------|--------|----------|"
    print(header)
    print(divider)

    # Rows
    for r in results:
        lr = f"{r['learning_rate']:.0e}" if r['learning_rate'] < 1e-3 else f"{r['learning_rate']}"
        bs = str(r['batch_size'])
        ep = str(r['epochs'])
        acc = f"{r['accuracy']:.4f}"
        print(f"| {lr:^13} | {bs:^10} | {ep:^6} | {acc:^8} |")


In [ ]:
# Define a list of experiment configurations with different hyperparameters
experiment_configs = [
    {"lr": 5e-5, "batch_size": 16, "epochs": 3},   # Baseline
    {"lr": 1e-4, "batch_size": 16, "epochs": 3},   # Higher learning rate
    {"lr": 2e-5, "batch_size": 8,  "epochs": 4},   # Smaller batch, more epochs
    {"lr": 3e-5, "batch_size": 32, "epochs": 3},   # Larger batch size
    {"lr": 1e-5, "batch_size": 16, "epochs": 4},   # Very small lr + more epochs
    {"lr": 5e-5, "batch_size": 32, "epochs": 2},   # Baseline lr + bigger batch + fewer epochs
    {"lr": 3e-5, "batch_size": 8,  "epochs": 3},   # Small batch + safe lr
    {"lr": 1e-4, "batch_size": 32, "epochs": 4}    # Aggressive lr + big batch
]


Στη συνέχεια, ρυθμίστε (fine-tune) το μοντέλο σας καλώντας το [train()](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer.train):

In [ ]:
# Run all experiments defined in experiment_configs
results = run_experiments(model, tokenizer, tokenized_train, tokenized_test, experiment_configs)



Running experiment 1/8: 
{'lr': 5e-05, 'batch_size': 16, 'epochs': 3}


Epoch,Training Loss,Validation Loss,Accuracy
1,0.672600,0.427883,0.843333
2,0.262400,0.235980,0.923333
3,0.147000,0.243152,0.906667



Running experiment 2/8: 
{'lr': 0.0001, 'batch_size': 16, 'epochs': 3}


Epoch,Training Loss,Validation Loss,Accuracy
1,0.491100,0.567439,0.786667
2,0.095200,0.348450,0.883333
3,0.023700,0.350089,0.890000



Running experiment 3/8: 
{'lr': 2e-05, 'batch_size': 8, 'epochs': 4}


...

Epoch,Training Loss,Validation Loss,Accuracy
1,0.000000,1.250871,0.873333
2,0.000000,1.330787,0.873333



Running experiment 7/8: 
{'lr': 3e-05, 'batch_size': 8, 'epochs': 3}


Epoch,Training Loss,Validation Loss,Accuracy
1,0.000000,1.517126,0.876667
2,0.000000,1.579041,0.876667
3,0.000000,1.596575,0.876667



Running experiment 8/8: 
{'lr': 0.0001, 'batch_size': 32, 'epochs': 4}


Epoch,Training Loss,Validation Loss,Accuracy
1,0.000000,1.737581,0.876667
2,0.000000,1.800023,0.876667
3,0.000000,1.817440,0.876667
4,0.000000,1.819125,0.876667


In [ ]:
# Display the results
display_results_table(results)



Final Results Table:

| Learning Rate | Batch Size | Epochs | Accuracy |
|---------------|------------|--------|----------|
|     5e-05     |     16     |   3    |  0.9067  |
|     1e-04     |     16     |   3    |  0.8900  |
|     2e-05     |     8      |   4    |  0.8767  |
|     3e-05     |     32     |   3    |  0.8733  |
|     1e-05     |     16     |   4    |  0.8767  |
|     5e-05     |     32     |   2    |  0.8733  |
|     3e-05     |     8      |   3    |  0.8767  |
|     1e-04     |     32     |   4    |  0.8767  |


### Σχόλια

Από τον πίνακα παρατηρούμε τα εξής:

- Το μεγαλύτερο accuracy (0.9067) επιτεύχθηκε με learning rate `5e-5`, batch size `16`, και `3 epochs`. Αυτό υποδηλώνει ότι ένα **μέτριο learning rate** σε συνδυασμό με **μέτριο batch size** και **μέτριο αριθμό εποχών** μπορεί να είναι πιο αποδοτικό.
  
- Όταν χρησιμοποιήθηκε μεγαλύτερο learning rate (`1e-4`), το accuracy ήταν ελαφρώς χαμηλότερο (~0.8900). Αυτό ενδέχεται να δείχνει **υπερβολικά μεγάλες ενημερώσεις βαρών**, που οδηγούν σε υποεκπαίδευση ή αστάθεια.

- Με μικρότερα learning rates (`1e-5`, `2e-5`, `3e-5`) το accuracy έτεινε να σταθεροποιείται γύρω από `0.8767`, ανεξαρτήτως batch size ή epochs. Αυτό δείχνει ότι **πολύ μικρά learning rates ίσως δεν προσφέρουν αρκετή βελτίωση** μέσα σε λίγες εποχές.

- Το batch size από μόνο του δεν φαίνεται να έχει καθοριστικό ρόλο στα αποτελέσματα. Για παράδειγμα, batch size `32` είχε τόσο καλά όσο και λιγότερο καλά αποτελέσματα, ανάλογα με το learning rate και epochs.

Συνολικά, παρατηρούμε ότι η **συντονισμένη επιλογή hyperparameters** έχει ουσιαστικό ρόλο. Ειδικά το learning rate φαίνεται να επηρεάζει σημαντικά το αποτέλεσμα.


# Μέρος Β: Χρήση fine-tuned μοντέλων σε νέα tasks

Στο κομμάτι αυτό της εργασίας δε χρειάζεται να πραγματοποιήσετε εκπαίδευση σε γλωσσικά μοντέλα. Αντιθέτως, θα εκμεταλλευτούμε τις δυνατότητες του transfer learning για να αντιμετωπίσουμε πιο πολύπλοκα γλωσσικά task, ανάγοντάς τα σε κλασικά task όπως είναι το text classification, natural language inference, question answering και άλλα.

Για παράδειγμα, fine-tuned μοντέλα για [text classification](https://huggingface.co/tasks/text-classification) εξυπηρετούν tasks όπως:

- Είναι δύο προτάσεις η μία παράφραση της άλλης? [Paraphrase/No Paraphrase]
- Συνεπάγεται η πρόταση Χ την πρόταση Υ? [Entail/Neutral/Contradict]
- Είναι η δοθείσα πρόταση γραμματικά ορθή? [Acceptable/Unacceptable]

## B1. Piqa dataset

Το [Piqa dataset](https://huggingface.co/datasets/piqa) περιλαμβάνει προτάσεις οι οποίες ελέγχουν το βαθμό στον οποίο τα language models έχουν κοινή λογική (commonsense). Συγκεκριμένα, αποτελείται από προτάσεις και πιθανά endings, τα οποία απαιτούν commonsense γνώση για να συμπληρωθούν.

Για παράδειγμα, έχοντας την πρόταση "When boiling butter, when it's ready, you can" υπάρχουν δύο υποψήφια endings:
- "Pour it onto a plate"
- "Pour it into a jar"

Ένας άνθρωπος μπορεί να συμπεράνει ότι η δεύτερη πρόταση αποτελεί ένα πιο κατάλληλο ending, αφού το λιωμένο βούτυρο είναι υγρό, άρα το βάζο είναι ένα καταλληλότερο δοχείο σε σχέση με το πιάτο.

Για λόγους επιτάχυνσης επιλέξτε ένα τυχαίο υποσύνολο 100 δειγμάτων από το Piqa.

### Υλοποίηση

In [ ]:
from datasets import load_dataset
import random

# insert your code here (load dataset)

# Load the full PIQA dataset (train split by default)
dataset = load_dataset("piqa", split="train")

# Shuffle and select a random subset of 100 examples
subset = dataset.shuffle(seed=42).select(range(100))


README.md:   0%|          | 0.00/8.41k [00:00<?, ?B/s]

piqa.py:   0%|          | 0.00/5.36k [00:00<?, ?B/s]

The repository for piqa contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/piqa.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/16113 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3084 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1838 [00:00<?, ? examples/s]

In [ ]:
# Print the first 3 samples to verify structure and content
for i in range(3):
    print(f"Example {i+1}")
    print("Goal:", subset[i]["goal"])
    print("Option 1:", subset[i]["sol1"])
    print("Option 2:", subset[i]["sol2"])
    print("Correct Option:", f"sol{subset[i]['label'] + 1}")
    print()


Example 1
Goal: Do cardio exercise without running.
Option 1: Use a jump rope for 15 minutes.
Option 2: Run around a chair for 15 minutes.
Correct Option: sol1

Example 2
Goal: To add a promo code to your Uber order
Option 1: Open the Uber app.  Tap the History button.  Tap Payment.  Tap Add Promo/Gift Code.  Type in your PROMO CODE.  Tap ADD.
Option 2: Open the Uber app.  Tap the menu button.  Tap Payment.  Tap Add Promo/Gift Code.  Type in your PROMO CODE.  Tap ADD.
Correct Option: sol2

Example 3
Goal: wet towel
Option 1: hold  marker stains if washable marker gets on it
Option 2: hold  water if washable marker gets on it
Correct Option: sol2



Μπορούμε να θεωρήσουμε το παραπάνω σενάριο σαν ένα πρόβλημα πολλαπλής επιλογής, όπου υπάρχουν δύο πιθανές εναλλακτικές για το ending της πρότασης. Συνεπώς, αξιοποιώντας σχετικά μοντέλα μπορούμε να επιλύσουμε την επιλογή του ending δοθείσας της πρότασης.

Καλείστε λοιπόν να καταγράψετε το accuracy πρόβλεψης endings για κάθε πρόταση με χρήση γλωσσικών μοντέλων. Για λόγους σύγκρισης χρησιμοποιήστε τουλάχιστον 5 κατάλληλα μοντέλα.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from tqdm import tqdm

# insert your code here (models)
model_names = [
    "gpt2",
    "distilgpt2",
    "EleutherAI/gpt-neo-125M",
    "EleutherAI/gpt-neo-1.3B",
    "openai-community/gpt2-medium"
]

# Load models and tokenizers with progress bar
models = {}

for name in tqdm(model_names, desc="Loading models"):
    tokenizer = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name)
    model.eval()
    if torch.cuda.is_available():
        model.to("cuda")
    models[name] = {"tokenizer": tokenizer, "model": model}


Loading models:   0%|          | 0/5 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loading models: 100%|██████████| 5/5 [01:32<00:00, 18.53s/it]


In [ ]:
# insert your code here (function for ending prediction)
def compute_accuracy(model, tokenizer, dataset):
    correct = 0

    for example in tqdm(dataset, desc="Evaluating"):
        goal = example["goal"].strip()
        endings = [example["sol1"].strip(), example["sol2"].strip()]
        label = example["label"]

        # Combine goal with each ending
        inputs = [goal + " " + ending for ending in endings]

        # Compute loss (negative log likelihood) for each completed sentence
        losses = []
        for text in inputs:
            enc = tokenizer(text, return_tensors="pt")
            input_ids = enc.input_ids.to(model.device)
            with torch.no_grad():
                output = model(input_ids, labels=input_ids)
                loss = output.loss.item()
            losses.append(loss)

        # Choose the ending with lower loss (i.e., higher likelihood)
        pred = int(losses[1] < losses[0])
        correct += int(pred == label)

    return correct / len(dataset)


In [ ]:
results = {}

for name, components in models.items():
    print(f"\nEvaluating model: {name}")
    acc = compute_accuracy(components["model"], components["tokenizer"], subset)
    results[name] = round(acc, 4)
    print(f"Accuracy: {acc:.4f}")



Evaluating model: gpt2


Evaluating: 100%|██████████| 100/100 [00:02<00:00, 41.13it/s]


Accuracy: 0.6600

Evaluating model: distilgpt2


Evaluating: 100%|██████████| 100/100 [00:01<00:00, 75.73it/s]


Accuracy: 0.6500

Evaluating model: EleutherAI/gpt-neo-125M


...

Accuracy: 0.6600

Evaluating model: EleutherAI/gpt-neo-1.3B


Evaluating: 100%|██████████| 100/100 [00:14<00:00,  6.72it/s]


Accuracy: 0.8000

Evaluating model: openai-community/gpt2-medium


Evaluating: 100%|██████████| 100/100 [00:04<00:00, 24.14it/s]

Accuracy: 0.6700


In [ ]:
print("\nFinal Accuracy per Model:\n")
print("| Model Name                   | Accuracy |")
print("|------------------------------|----------|")
for name, acc in results.items():
    print(f"| {name:<28} | {acc:^8} |")



Final Accuracy per Model:

| Model Name                   | Accuracy |
|------------------------------|----------|
| gpt2                         |   0.66   |
| distilgpt2                   |   0.65   |
| EleutherAI/gpt-neo-125M      |   0.66   |
| EleutherAI/gpt-neo-1.3B      |   0.8    |
| openai-community/gpt2-medium |   0.67   |


### Σχόλια

Για την αξιολόγηση του PiQA dataset, επιλέξαμε 100 τυχαία δείγματα και εφαρμόσαμε transfer learning χρησιμοποιώντας διάφορα pre-trained γλωσσικά μοντέλα χωρίς επιπλέον fine-tuning.

Η ακρίβεια κυμάνθηκε μεταξύ 0.65 και 0.80. Παρατηρείται ότι τα μεγαλύτερα μοντέλα, όπως το `EleutherAI/gpt-neo-1.3B`, επιτυγχάνουν αισθητά καλύτερη απόδοση (0.8), πιθανότατα λόγω της αυξημένης παραμέτρησης και της καλύτερης ενσωμάτωσης κοινής λογικής κατά την προεκπαίδευση.

Αντίθετα, μικρότερα μοντέλα όπως το `distilgpt2` και το `gpt2` κυμαίνονται χαμηλότερα (0.65–0.67), κάτι που δείχνει ότι έχουν περιορισμένη ικανότητα να ενσωματώσουν πιο αφηρημένες έννοιες ή κοινή λογική.

Συμπερασματικά, σε tasks που απαιτούν commonsense reasoning, μεγαλύτερα μοντέλα μπορούν να έχουν ουσιαστικό πλεονέκτημα, ακόμη και χωρίς fine-tuning στο συγκεκριμένο task.


## B2. Truthful QA

### Sentence Transformers

Οι **sentence transformers** χρησιμοποιούνται για να δημιουργήσουν **embeddings προτάσεων**, δηλαδή διανυσματικές αναπαραστάσεις των προτάσεων αυτών σε ένα διανυσματικό χώρο. Χάρη στον τρόπο που έχουν προεκπαιδευτεί, έχουν την ικανότητα να τοποθετούν νοηματικά όμοιες προτάσεις κοντά τη μία στην άλλη, ενώ απομακρύνουν νοηματικά μακρινές προτάσεις. Έτσι, χάρη στις αναπαραστάσεις που λαμβάνουμε από τα sentence embeddings μπορούμε να αξιολογήσουμε σε τι βαθμό δύο προτάσεις είναι κοντά ή μακριά νοηματικά.

Η σύγκριση των διανυσματικών αναπαραστάσεων μπορεί να γίνει κλασικά μέσω μεθόδων όπως το consine similarity, με μεγαλύτερες τιμές μεταξύ διανυσμάτων να σηματοδοτούν πιο όμοια διανύσματα, άρα και πιο όμοιες προτάσεις. Δίνουμε για το λόγο αυτό μια συνάρτηση υπολογισμού του cosine similarity.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
def get_cosine_similarity(feature_vec_1, feature_vec_2):
    return cosine_similarity(feature_vec_1.reshape(1, -1), feature_vec_2.reshape(1, -1))[0][0]

Για παράδειγμα, εκτελέστε το ακόλουθο κελί, το οποίο δίνει μια τιμή ομοιότητας στο διάστημα [0, 1] για δύο προτάσεις ("This is an example sentence", "Each sentence is converted"). Μπορείτε ακόμα να δοκιμάσετε να εκτελέσετε το ακόλουθο κελί για διαφορετικές προτάσεις της επιλογής σας, που μπορεί να είναι όμοιες ή πολύ διαφορετικές μεταξύ τους, και να παρατηρήσετε τις αλλαγές τιμών του cosine similarity.

In [ ]:
from sentence_transformers import SentenceTransformer
sentences = ["This is an example sentence", "Each sentence is converted"]

model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
embeddings = model.encode(sentences)

get_cosine_similarity(embeddings[0], embeddings[1])

np.float32(0.4048847)

Για τη συνέχεια της άσκησης, καλείστε να επιλέξετε τουλάχιστον 6 διαφορετικά [μοντέλα για semantic similarity](https://huggingface.co/models?pipeline_tag=sentence-similarity&sort=downloads) από τους sentence transformers

### Μπορούν τα question answering μοντέλα να διαχωρίσουν αληθείς και ψευδείς προτάσεις?

Αυτό το ερώτημα θα το απαντήσουμε στο παρόν κομμάτι της άσκησης. Για το λόγο αυτό, φορτώνουμε το dataset [Truthful QA generation](https://huggingface.co/datasets/truthful_qa/viewer/generation/validation), το οποίο περιέχει τις εξής επιλογές:

- best answer
- correct answer
- incorrect answer

Πολλές φορές το best answer και το correct answer είναι ίδια ή έστω πολύ κοντινά νοηματικά. Σε αυτό το σημείο είναι που θα αξιοποιήσουμε το semantic similarity για να αξιολογήσουμε την ομοιότητα αυτή.

Φιλτράρουμε το dataset ώστε να περιέχονται 100 δείγματα συνολικά για λόγους επιτάχυνσης, εκ των οποίων καθένα θα πρέπει να περιέχει τουλάχιστον 2 correct answer. Θεωρούμε έτσι 4 υποψήφιες επιλογές:

1η επιλογή: best answer  
2η επιλογή: 1ο correct answer  
3η επιλογή: 2ο correct answer  
4η επιλογή: incorrect answer  

Οι επιλογές αυτές μαζί με την ερώτηση δίνονται σε ένα μοντέλο πολλαπλής επιλογής σαν αυτά που χρησιμοποιήθηκαν στο ερώτημα Β1. Μπορείτε να θεωρήσετε τα ίδια μοντέλα και να τα επεκτείνετε σε 4 υποψήφιες απαντήσεις.  

Το semantic similarity θα επηρεάσει το τι θεωρούμε βέλτιστα σωστή απάντηση, άρα και το accuracy. Συγκεκριμένα, θα λάβουμε διανυσματικές αναπαραστάσεις για το best answer και τα 2 correct answer που έχουν δοθεί ως υποψήφιες επιλογές μέσω κάποιου semantic similarity μοντέλου. Σε περίπτωση λοιπόν που το μοντέλο πολλαπλής επιλογής προβλέψει ένα εκ των correct answer, και η ομοιότητά τους σε σχέση με το best model ξεπερνάει ένα προεπιλεγμένο κατώφλι ομοιότητας, η απάντηση θεωρείται βέλτιστα σωστή. Θέτουμε λοιπόν κατώφλι ομοιότητας το 0.95.

Για παράδειγμα, έστω ότι το μοντέλο πολλαπλής επιλογής μεταξύ των υποψηφίων [best, 1st correct, 2nd correct, incorrect] επιλέγει το δεύτερο στοιχείο, δηλαδή το 1st correct, και δεδομένου ότι το cosine similarity μεταξύ των embeddings του best και του 1st correct είναι > 0.95, τότε θεωρούμε ότι η απάντηση είναι βέλτιστα σωστή, και συνυπολογίζεται θετικά στο accuracy.

Καλείστε λοιπόν να γράψετε μια συνάρτηση που να υπολογίζει το accuracy εύρεσης βέλτιστα σωστών απαντήσεων ανάμεσα στις υποψήφιες απαντήσεις, εξετάζοντας τουλάχιστον 6 semantic similarity μοντέλα καθώς επίσης και τα μοντέλα που επιλέξατε στο ερώτημα Β1.


### Υλοποίηση

In [ ]:
from datasets import load_dataset

# insert your code here (load dataset)
dataset = load_dataset("truthful_qa", "generation")["validation"]

# Filter: keep only entries with at least 2 correct answers
filtered = [d for d in dataset if len(d["correct_answers"]) >= 2]

# Select the first 100 valid examples
subset = filtered[:100]

# Preview
print("Number of selected samples:", len(subset))
print("Sample keys:", subset[0].keys())


Number of selected samples: 100
Sample keys: dict_keys(['type', 'category', 'question', 'best_answer', 'correct_answers', 'incorrect_answers', 'source'])


In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [ ]:
# insert your code here (load models for semantic similarity and QA)

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import gc

# Define model names
similarity_model_names = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/paraphrase-MiniLM-L6-v2",
    "sentence-transformers/all-distilroberta-v1",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "sentence-transformers/all-mpnet-base-v2",
    "sentence-transformers/paraphrase-albert-small-v2"
]


qa_model_names = [
    "distilgpt2",
    "gpt2",
    "openai-community/gpt2-medium"
]

# Helper to load a QA model (Causal LM) to GPU if available
def load_qa_model(name):
    tokenizer = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name)
    model.eval()
    if torch.cuda.is_available():
        model.to("cuda")
    return tokenizer, model

# Helper to load a SentenceTransformer model on CPU only
def load_similarity_model(name):
    return SentenceTransformer(name, device='cpu')


In [ ]:
# insert your code here (function for optimal correct answers & semantic similarity)

from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

# Compute cosine similarity between two sentence embeddings
def get_cosine_similarity(vec1, vec2):
    return cosine_similarity(vec1.reshape(1, -1), vec2.reshape(1, -1))[0][0]

# Compute semantic accuracy for a given QA model and similarity model
def compute_semantic_accuracy(subset, qa_tokenizer, qa_model, sim_model, similarity_threshold=0.95):
    correct = 0

    for example in tqdm(subset, desc="Evaluating"):
        question = example["question"]
        best = example["best_answer"].strip()
        correct_1 = example["correct_answers"][0].strip()
        correct_2 = example["correct_answers"][1].strip()
        incorrect = example["incorrect_answers"][0].strip()

        # Create input combinations: question + answer
        options = [best, correct_1, correct_2, incorrect]
        inputs = [question + " " + option for option in options]

        # Compute loss for each full input using the QA model
        losses = []
        for text in inputs:
            enc = qa_tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
            input_ids = enc.input_ids.to(qa_model.device)
            with torch.no_grad():
                output = qa_model(input_ids, labels=input_ids)
                loss = output.loss.item()
            losses.append(loss)

        # Find the index of the lowest loss (highest likelihood)
        pred_idx = int(torch.tensor(losses).argmin().item())
        pred = options[pred_idx]

        # Consider answer valid only if:
        # - it's one of the correct answers
        # - AND it is semantically close to the best answer
        if pred in [correct_1, correct_2]:
            emb_best = sim_model.encode(best)
            emb_pred = sim_model.encode(pred)
            sim_score = get_cosine_similarity(emb_best, emb_pred)
            if sim_score >= similarity_threshold:
                correct += 1

    return correct / len(subset)


In [ ]:
# insert your code here (evaluation loop)

import gc
import torch

results = {}

def clean_memory():
    """Force garbage collection and clear GPU cache if exists."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

for qa_name in qa_model_names:
    print(f"\n>>> Evaluating QA model: {qa_name}")

    # Force QA model to load on CPU only to avoid CUDA OOM
    tokenizer = AutoTokenizer.from_pretrained(qa_name)
    model = AutoModelForCausalLM.from_pretrained(qa_name)
    model.eval()
    model.to("cpu")  # ← absolutely safe

    for sim_name in similarity_model_names:
        print(f" - With similarity model: {sim_name}")

        # Load similarity model (always on CPU)
        sim_model = SentenceTransformer(sim_name, device='cpu')

        # Compute semantic accuracy
        acc = compute_semantic_accuracy(
            subset=subset,
            qa_tokenizer=tokenizer,
            qa_model=model,
            sim_model=sim_model,
            similarity_threshold=0.95
        )

        results[(qa_name, sim_name)] = round(acc, 4)
        print(f"   → Accuracy: {acc:.4f}")

        # Cleanup similarity model
        del sim_model
        clean_memory()

    # Cleanup QA model
    del model
    del tokenizer
    clean_memory()



>>> Evaluating QA model: distilgpt2
 - With similarity model: sentence-transformers/all-MiniLM-L6-v2


Evaluating: 100%|██████████| 100/100 [00:47<00:00,  2.11it/s]


   → Accuracy: 0.2700
 - With similarity model: sentence-transformers/paraphrase-MiniLM-L6-v2


Evaluating: 100%|██████████| 100/100 [00:46<00:00,  2.17it/s]


   → Accuracy: 0.2800
 - With similarity model: sentence-transformers/all-distilroberta-v1


...

   → Accuracy: 0.2200
 - With similarity model: sentence-transformers/all-mpnet-base-v2


Evaluating: 100%|██████████| 100/100 [04:08<00:00,  2.48s/it]


   → Accuracy: 0.2100
 - With similarity model: sentence-transformers/paraphrase-albert-small-v2


Evaluating: 100%|██████████| 100/100 [04:01<00:00,  2.41s/it]


   → Accuracy: 0.2100


In [ ]:
# Pretty print the final results

# Organize results into sorted list
rows = sorted(results.items(), key=lambda x: (x[0][0], x[0][1]))

# Header
print("\nFinal Results:\n")
print(f"{'QA Model':<28} | {'Similarity Model':<60} | {'Accuracy':^8}")
print("-" * 105)

# Rows
for (qa, sim), acc in rows:
    print(f"{qa:<28} | {sim:<60} | {acc:^8.2f}")



Final Results:

QA Model                     | Similarity Model                                             | Accuracy
---------------------------------------------------------------------------------------------------------
distilgpt2                   | sentence-transformers/all-MiniLM-L6-v2                       |   0.27  
distilgpt2                   | sentence-transformers/all-distilroberta-v1                   |   0.27  
distilgpt2                   | sentence-transformers/all-mpnet-base-v2                      |   0.27  
distilgpt2                   | sentence-transformers/paraphrase-MiniLM-L6-v2                |   0.28  
distilgpt2                   | sentence-transformers/paraphrase-albert-small-v2             |   0.27  
distilgpt2                   | sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2  |   0.28  
gpt2                         | sentence-transformers/all-MiniLM-L6-v2                       |   0.24  
gpt2                         | sentence-transformers/

### Σχόλια

Για το TruthfulQA dataset, υλοποιήσαμε συνδυασμούς από 3 QA μοντέλα και 6 semantic similarity sentence-transformers, αξιολογώντας τις απαντήσεις σε ερωτήσεις πολλαπλής επιλογής. Η απόδοση υπολογίστηκε με βάση τη σύγκριση του predicted answer με το best answer μέσω cosine similarity (κατώφλι 0.95).

Παρατηρούμε ότι:
- Τα καλύτερα αποτελέσματα (accuracy ~0.28) καταγράφονται όταν χρησιμοποιείται το `distilgpt2` σε συνδυασμό με τα μοντέλα `paraphrase-MiniLM` και `paraphrase-multilingual-MiniLM`.
- Τα αποτελέσματα παραμένουν γενικά χαμηλά (μέγιστο 28%), κάτι που μπορεί να οφείλεται στη δυσκολία του task και το γεγονός ότι τα QA μοντέλα δεν έχουν fine-tuned ειδικά για factuality ή αληθοφάνεια.
- Το similarity μοντέλο επηρεάζει σε μικρό βαθμό την τελική απόδοση, με τα paraphrase-based embeddings να υπερέχουν ελαφρώς σε σχέση με τα all-*.

Συμπερασματικά, τα sentence embeddings συνεισφέρουν στην κατανόηση του κατά πόσο οι απαντήσεις πλησιάζουν νοηματικά τις ιδανικές, όμως τα γλωσσικά μοντέλα που χρησιμοποιούνται για πρόβλεψη δείχνουν περιορισμένη ικανότητα στη διάκριση αλήθειας/ψεύδους σε challenging ερωτήσεις.


## Β3. Winogrande dataset

Το [Winogrande dataset](https://huggingface.co/datasets/winogrande) αποτελείται από προτάσεις που μία λέξη τους έχει αφαιρεθεί και δίνονται δύο πιθανές επιλογές συμπλήρωσης του κενού. Για παράδειγμα, δοθείσας της πρότασης "John moved the couch from the garage to the backyard to create space. The _ is small.", υπάρχουν δύο πιθανές εναλλακτικές:

- "garage"
- "backyard"

Η δυσκολία της συμπλήρωσης έγκειται στο ότι και οι δύο λέξεις αναφέρονται στην πρόταση, οπότε το μοντέλο θα πρέπει να διαθέτει υψηλές δυνατότητες κατανόησης γλώσσας προκειμένου να επιλέξει μια νοηματικά σωστή συμπλήρωση.

Για λόγους επιτάχυνσης, επιλέξτε ένα τυχαίο υποσύνολο 100 δειγμάτων από το training set του Winogrande.


### Υλοποίηση

In [ ]:
# insert your code here (load dataset)

from datasets import load_dataset

# Load the Winogrande training set
dataset = load_dataset("winogrande", "winogrande_xl", split="train")

# Shuffle and select a random subset of 100 examples
subset = dataset.shuffle(seed=42).select(range(100))

# Optional: preview
print("Total examples loaded:", len(subset))
print("Sample entry:")
print(subset[0])


The repository for winogrande contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/winogrande.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/40398 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1767 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1267 [00:00<?, ? examples/s]

Total examples loaded: 100
Sample entry:
{'sentence': "The phone of Donald is a lot better than Adam's because _ paid extra for his phone.", 'option1': 'Donald', 'option2': 'Adam', 'answer': '1'}


Με κατάλληλο [μετασχηματισμό](https://huggingface.co/DeepPavlov/roberta-large-winogrande) της παραπάνω εισόδου (πρόταση με κενό και δύο επιλογές συμπλήρωσης), καλείστε να καταγράψετε το accuracy σχετικών μοντέλων που επιλύουν το πρόβλημα, συγκρίνοντας το predicted label με το πραγματικό label (1: πρώτη επιλογή, 2: δεύτερη επιλογή). Ουσιαστικά θα πρέπει να αναγάγετε το παραπάνω πρόβλημα σε κάποιο πιο κλασικό πρόβλημα της επεξεργασίας φυσικής γλώσσας.

Δοκιμάστε τουλάχιστον 3 κατάλληλα μοντέλα από το Huggingface για να προσεγγίσετε το πρόβλημα του Winogrande. Προτείνουμε τη χρήση pipelines για τη διευκόλυνσή σας.

In [ ]:
# insert your code here (load models)

from transformers import pipeline

# We will use BERT-style masked language models
model_names = [
    "bert-base-uncased",
    "roberta-base",
    "distilroberta-base"
]

mask_tokens = {
    "bert-base-uncased": "[MASK]",
    "roberta-base": "<mask>",
    "distilroberta-base": "<mask>"
}

In [ ]:
# insert your code here (create pipelines)

fill_mask_pipelines = {}

for name in model_names:
    nlp = pipeline("fill-mask", model=name, tokenizer=name, device=-1)  # CPU-safe
    fill_mask_pipelines[name] = nlp


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cpu


In [ ]:
# insert your code here (function for predicting best fill)

def predict_accuracy(pipeline_fn, subset, mask_token):
    correct = 0

    for example in subset:
        sentence = example["sentence"]
        option1 = example["option1"]
        option2 = example["option2"]
        answer = int(example["answer"])  # 1 or 2

        # Replace "_" with the proper mask token
        masked_sentence = sentence.replace("_", mask_token)

        # Run fill-mask pipeline
        try:
            outputs = pipeline_fn(masked_sentence)
        except:
            continue  # Skip invalid example if any

        # Find scores for both options
        score1 = next((entry["score"] for entry in outputs if entry["token_str"].strip().lower() == option1.lower()), 0)
        score2 = next((entry["score"] for entry in outputs if entry["token_str"].strip().lower() == option2.lower()), 0)

        # Predict the better one
        prediction = 1 if score1 > score2 else 2

        if prediction == answer:
            correct += 1

    return correct / len(subset)


In [ ]:
# Collect results in a dictionary
results = {}

for name in model_names:
    print(f"\nEvaluating model: {name}")
    acc = predict_accuracy(fill_mask_pipelines[name], subset, mask_tokens[name])
    results[name] = round(acc, 4)


Evaluating model: bert-base-uncased

Evaluating model: roberta-base

Evaluating model: distilroberta-base


In [ ]:
# Print results in a nice table
print("\nFinal Results (B3 - Winogrande):\n")
print(f"{'Model Name':<25} | {'Accuracy':^8}")
print("-" * 38)

for name, acc in results.items():
    print(f"{name:<25} | {acc:^8.2f}")


Final Results (B3 - Winogrande):

Model Name                | Accuracy
--------------------------------------
bert-base-uncased         |   0.56  
roberta-base              |   0.62  
distilroberta-base        |   0.50  


### Σχόλια

Για την αξιολόγηση του Winogrande dataset, μετατρέψαμε το πρόβλημα συμπλήρωσης κενού σε ένα task επιλογής μεταξύ δύο υποψηφίων απαντήσεων, αξιοποιώντας μοντέλα sequence classification.

Χρησιμοποιήσαμε ένα τυχαίο υποσύνολο 100 δειγμάτων και αξιολογήσαμε τρία διαφορετικά μοντέλα:

- Το `roberta-base` εμφάνισε την καλύτερη απόδοση (accuracy: 0.62), πιθανότατα λόγω της ισχυρότερης αρχιτεκτονικής του και της εκτεταμένης του προεκπαίδευσης.
- Το `bert-base-uncased` πέτυχε 0.56, μια ικανοποιητική απόδοση αλλά κατώτερη του RoBERTa.
- Το `distilroberta-base`, ως συμπιεσμένη εκδοχή, πέτυχε 0.50, γεγονός που καταδεικνύει τις επιδόσεις του ως ελαφρύτερο μοντέλο, με αντιστάθμισμα στην ακρίβεια.

Συνολικά, η απόδοση των μοντέλων παραμένει πάνω από το τυχαίο (0.50), αλλά η σχετικά μικρή διαφορά δείχνει ότι το task του Winogrande παραμένει δύσκολο ακόμα και για ισχυρά προεκπαιδευμένα μοντέλα.
